In [1]:
import json
import os
import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity


def ket0():
    return np.array([1.0, 0.0], dtype=complex)


def ket1():
    return np.array([0.0, 1.0], dtype=complex)


def projector(v):
    return np.outer(v, v.conj())


def main():
    os.makedirs("results", exist_ok=True)

    # Build two Bell pairs: (q0,q1) and (q2,q3)
    qc = QuantumCircuit(4)
    qc.h(0)
    qc.cx(0, 1)
    qc.h(2)
    qc.cx(2, 3)

    psi = Statevector.from_instruction(qc).data

    # Bell state |Phi+>
    phi = (np.kron(ket0(), ket0()) + np.kron(ket1(), ket1())) / np.sqrt(2.0)
    bell_proj = projector(phi)

    # Project q1,q2 onto |Phi+>
    P12 = (
        np.kron(projector(ket0()), np.kron(bell_proj, projector(ket0())))
        + np.kron(projector(ket0()), np.kron(bell_proj, projector(ket1())))
        + np.kron(projector(ket1()), np.kron(bell_proj, projector(ket0())))
        + np.kron(projector(ket1()), np.kron(bell_proj, projector(ket1())))
    )

    psi_post_unnorm = P12 @ psi
    success_probability = float(np.vdot(psi_post_unnorm, psi_post_unnorm).real)

    if success_probability < 1e-15:
        raise RuntimeError("Postselected branch has zero probability.")

    psi_post = psi_post_unnorm / np.sqrt(success_probability)
    rho_post = DensityMatrix(np.outer(psi_post, psi_post.conj()))

    # Trace out q1 and q2, keep q0 and q3
    rho_03 = partial_trace(rho_post, [1, 2])

    target_rho = DensityMatrix(np.outer(phi, phi.conj()))
    bell_fidelity = float(state_fidelity(rho_03, target_rho))
    entanglement_yield = success_probability * bell_fidelity

    result = {
        "stack": "qiskit",
        "paradigm": "DV",
        "protocol": "two_bellpair_entanglement_swapping",
        "success_probability": success_probability,
        "bell_fidelity": bell_fidelity,
        "entanglement_yield": entanglement_yield,
        "notes": "Qiskit local statevector reference with analytic postselection on |Phi+>_12."
    }

    with open("results/qiskit.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()

{
  "stack": "qiskit",
  "paradigm": "DV",
  "protocol": "two_bellpair_entanglement_swapping",
  "success_probability": 0.24999999999999978,
  "bell_fidelity": 0.9999999999999991,
  "entanglement_yield": 0.24999999999999956,
  "notes": "Qiskit local statevector reference with analytic postselection on |Phi+>_12."
}
